# 02 — XML reading, writing, and CLI equivalents


This notebook shows how to write a network to XML, read it back, and use the command-line tools. The XML reader is permissive and is intended to handle compact `nucnetpy` XML plus many NucNet-style files.

In [ ]:
from pathlib import Path
import sys

# Make the local editable package importable when the notebooks are run from the repo.
HERE = Path.cwd()
CANDIDATES = [HERE / "src", HERE.parent / "src", HERE.parent.parent / "src"]
for p in CANDIDATES:
    if (p / "nucnetpy").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn
print("nucnetpy version:", nn.__version__)
def build_toy_alpha_network():
    """Build a small alpha-chain toy network for tutorials.

    This is not intended to be a physical helium-burning network.  The rates are
    deliberately simple constants so that the examples run quickly and are easy
    to inspect.
    """
    net = nn.Network()
    for name in ["he4", "be8", "c12"]:
        net.add_species(nn.Species.parse(name))

    r1 = nn.Reaction.from_names(
        ["he4", "he4"], ["be8"],
        constant_rate=2.0e-6,
        q_value=0.092,
        label="toy_2alpha_to_be8",
    )
    r2 = nn.Reaction.from_names(
        ["be8", "he4"], ["c12"],
        constant_rate=5.0e-5,
        q_value=7.367,
        label="toy_be8_alpha_to_c12",
    )
    net.reactions.add(r1)
    net.reactions.add(r2)

    zone = nn.Zone(label=("toy", "0", "0"), properties={"t9": "1.0", "rho": "1e4"})
    zone.set_abundance("he4", 0.25)  # X(he4) = A*Y = 1.0
    zone.set_abundance("be8", 0.0)
    zone.set_abundance("c12", 0.0)
    net.add_zone(zone)
    return net

In [ ]:
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
xml_path = data_dir / 'toy_alpha_network.xml'
net = build_toy_alpha_network()
nn.write_xml(net, xml_path)
print(xml_path.resolve())
print(xml_path.read_text()[:800])

## Read the XML file back

In [ ]:
net2 = nn.read_xml(xml_path)
print('species:', net2.species_names())
print('zones:', len(net2.zones))
for r in net2.reactions.reactions:
    print(r.string, 'rate at T9=1:', r.rate(1.0))

## Basic CLI commands

After installing the package, you can call `nucnetpy` from the terminal. In a notebook, the same commands can be run with `!`.

In [ ]:
import nucnetpy.cli as cli
cli.main(['summary', 'data/toy_alpha_network.xml'])

In [ ]:
cli.main(['rates', 'data/toy_alpha_network.xml', '--t9', '1.0', '--rho', '1e4'])

In [ ]:
cli.main(['conservation', 'data/toy_alpha_network.xml'])